Import Libraries

In [12]:
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

Load Dataset

In [13]:
dataset = load_dataset("Tobi-Bueck/customer-support-tickets")
df = dataset['train'].to_pandas()
print(df.head())

                                             subject  \
0                    Wesentlicher Sicherheitsvorfall   
1                                 Account Disruption   
2  Query About Smart Home System Integration Feat...   
3                  Inquiry Regarding Invoice Details   
4  Question About Marketing Agency Software Compa...   

                                                body  \
0  Sehr geehrtes Support-Team,\n\nich möchte eine...   
1  Dear Customer Support Team,\n\nI am writing to...   
2  Dear Customer Support Team,\n\nI hope this mes...   
3  Dear Customer Support Team,\n\nI hope this mes...   
4  Dear Support Team,\n\nI hope this message reac...   

                                              answer      type  \
0  Vielen Dank für die Meldung des kritischen Sic...  Incident   
1  Thank you for reaching out, <name>. We are awa...  Incident   
2  Thank you for your inquiry. Our products suppo...   Request   
3  We appreciate you reaching out with your billi...   Request

In [14]:
df.isnull().sum()

,0
subject,5299
body,2
answer,13189
type,13178
queue,0
priority,0
language,0
version,33178
tag_1,13178
tag_2,13237


In [15]:
df['subject'] = df['subject'].fillna("")
df['body'] = df['body'].fillna("")

df['text'] = df['subject'] + " " + df['body']
df=df[df["text"].str.len()>0].copy()


In [16]:

le = LabelEncoder()
df["label"] = le.fit_transform(df["queue"])

In [17]:
label_names=list(le.classes_)
num_labels=len(label_names)
print(num_labels)
print(label_names)

52
['Arts & Entertainment/Movies', 'Arts & Entertainment/Music', 'Autos & Vehicles/Maintenance', 'Autos & Vehicles/Sales', 'Beauty & Fitness/Cosmetics', 'Beauty & Fitness/Fitness Training', 'Billing and Payments', 'Books & Literature/Fiction', 'Books & Literature/Non-Fiction', 'Business & Industrial/Manufacturing', 'Customer Service', 'Finance/Investments', 'Finance/Personal Finance', 'Food & Drink/Groceries', 'Food & Drink/Restaurants', 'Games', 'General Inquiry', 'Health/Medical Services', 'Health/Mental Health', 'Hobbies & Leisure/Collectibles', 'Hobbies & Leisure/Crafts', 'Home & Garden/Home Improvement', 'Home & Garden/Landscaping', 'Human Resources', 'IT & Technology/Hardware Support', 'IT & Technology/Network Infrastructure', 'IT & Technology/Security Operations', 'IT & Technology/Software Development', 'IT Support', 'Jobs & Education/Online Courses', 'Jobs & Education/Recruitment', 'Law & Government/Government Services', 'Law & Government/Legal Advice', 'News', 'Online Communit

In [18]:

#Train,validation,test splits
train_df,temp_df=train_test_split(df,test_size=0.2,random_state=42,stratify=df["label"])
val_df,test_df=train_test_split(temp_df,test_size=0.5,random_state=42,stratify=temp_df["label"])

In [19]:

train_ds=Dataset.from_pandas(train_df[["text","label"]].reset_index(drop=True))
val_ds=Dataset.from_pandas(val_df[["text","label"]].reset_index(drop=True))
test_ds=Dataset.from_pandas(test_df[["text","label"]].reset_index(drop=True))

In [20]:
dataset=DatasetDict({
    "train":train_ds,
    "validation":val_ds,
    "test":test_ds,
})

In [21]:
model_checkpoint="bert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [22]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",   # ADD THIS
        max_length=256
    )

In [23]:
tokenized_dataset=dataset.map(tokenize_batch,batched=True,remove_columns=["text"])
tokenized_dataset.set_format("torch")
id2label={i:l for i,l in enumerate(label_names)}
label2id={l:i for i,l in enumerate(label_names)}

Map:   0%|          | 0/49412 [00:00<?, ? examples/s]

Map:   0%|          | 0/6176 [00:00<?, ? examples/s]

Map:   0%|          | 0/6177 [00:00<?, ? examples/s]

In [28]:
from transformers import DataCollatorWithPadding

model=AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
data_collator=DataCollatorWithPadding(tokenizer=tokenizer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [29]:

#metrics
def compute_metrics(eval_pred):
  predictions,labels=eval_pred
  predictions=np.argmax(predictions,axis=1)
  return {"macro_f1":f1_score(labels,predictions,average="macro"),"acc":accuracy_score(labels,predictions),
          "weighted_f1":f1_score(labels,predictions,average="weighted")}

In [31]:
import torch

training_args=TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
)

In [32]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:

trainer.evaluate(tokenized_dataset["test"])

In [ ]:
def predict(text):
  model.eval()
  inputs=tokenizer(text,return_tensors="pt",truncation=True,max_length=256)
  inputs={k:v.to(model.device) for k,v in inputs.items()}
  with torch.no_grad():
    logits=model(**inputs).logits
  predictions=torch.argmax(logits,dim=-1).cpu().numpy()
  return le.inverse_transform([predictions])[0]